# 🔁 Notebook 1: Retry Strategies — Bad → Best

Networks flap. Servers restart. A call that fails once often succeeds on the
very next try. **Retrying** is one of the simplest resilience tools we have.

But a naive retry can make things **worse** — causing outages, duplicate
payments, and retry storms. In this notebook we walk through four strategies,
from worst to best, and see exactly **why** each next step is an improvement.

1. ❌ **Immediate retry** — try again right away.
2. 🟡 **Fixed delay** — wait N seconds, try again.
3. 🟢 **Exponential backoff** — wait 1, 2, 4, 8... seconds.
4. ✅ **Exponential + jitter** — backoff with a random ± component.

> ⚠️ Only retry **idempotent** operations (see `04-patterns/idempotency`).
> Retrying a non-idempotent call like `POST /charge` without an idempotency
> key can charge a customer twice.

## 🛠️ Setup

```bash
cd 05-microservices/retry
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## A flaky dependency

We'll simulate a downstream service that fails 60% of the time with a
transient error. Think: momentary network blip, brief DB lock, leader
election, cold-start.

In [ ]:
import time, random

def flaky(p_fail=0.6):
    """Simulated transient failure."""
    if random.random() < p_fail:
        raise RuntimeError('transient boom')
    return 'ok'


## ❌ Bad: Immediate retry (no wait at all)

Why it's bad:
- If the service is overloaded, slamming it with instant retries makes it *worse*.
- If 1000 clients all do this, you multiply traffic by your retry count —
  a great way to DDoS your own backend.
- No `max_attempts` or deadline → you can loop forever on a hard failure.

In [ ]:
def retry_immediate_bad(fn, **kw):
    # ⚠️ demo only — no backoff, no cap, retries on ANY exception
    while True:
        try:
            return fn(**kw)
        except Exception:
            pass  # hammer the service

random.seed(1)
print(retry_immediate_bad(flaky, p_fail=0.7))  # succeeds eventually — but at what cost?


## A safer retry skeleton

Before showing the other strategies, let's build a reusable helper with
the essentials every production retry needs:

- A **max attempt count** (stop hammering a dead service).
- A pluggable **delay strategy** (fixed / exponential / jitter).
- A **visible log** so we can see what happened.

In [ ]:
def retry(fn, strategy, max_attempts=6, **kw):
    """Call `fn(**kw)`. On failure, sleep `strategy(attempt)` seconds and try again."""
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(**kw), attempt
        except Exception as e:
            delay = strategy(attempt)
            print(f'  attempt {attempt} failed ({e}) → sleep {delay:.2f}s')
            time.sleep(delay)
    raise RuntimeError(f'gave up after {max_attempts} attempts')


## 🟡 Better: Fixed delay

At least we stop hammering. Each client waits the same amount after every
failure. Problem: if everyone uses the same delay, everyone retries at the
**same moment** (see Notebook 2 — *thundering herd*).

In [ ]:
random.seed(2)
result, n = retry(flaky, strategy=lambda a: 0.2, max_attempts=8, p_fail=0.7)
print(f'succeeded on attempt {n} → {result}')


## 🟢 Good: Exponential backoff

Double the wait each time: 0.1s, 0.2s, 0.4s, 0.8s... This gives the
downstream room to recover. The idea: *"if it's still broken, we probably
need to wait a lot longer."*

Still not perfect — multiple clients synchronized on the same failure will
still retry together.

In [ ]:
def exponential(base=0.1, factor=2.0, cap=5.0):
    """Return a strategy function. Caps the delay so it doesn't grow forever."""
    return lambda a: min(cap, base * (factor ** (a - 1)))

random.seed(3)
result, n = retry(flaky, strategy=exponential(), max_attempts=8, p_fail=0.7)
print(f'succeeded on attempt {n} → {result}')


## ✅ Best: Exponential backoff + jitter

Multiply the backoff by a random factor (e.g. uniform 0.5–1.5, or full
random from 0 to the cap). Clients now retry at **different times**, so the
service isn't hit by synchronized waves.

This is the AWS-recommended default — see the classic blog post
*"Exponential Backoff And Jitter"* (linked in the README).

In [ ]:
def exponential_jitter(base=0.1, factor=2.0, cap=5.0):
    """`Full jitter`: pick a random delay in [0, exp_backoff]."""
    def strat(a):
        exp = min(cap, base * (factor ** (a - 1)))
        return random.uniform(0, exp)
    return strat

random.seed(4)
result, n = retry(flaky, strategy=exponential_jitter(), max_attempts=8, p_fail=0.7)
print(f'succeeded on attempt {n} → {result}')


## Four strategies side-by-side

Run them on the same (very flaky — 70%) dependency and watch the delays.

In [ ]:
strategies = {
    'immediate':   lambda a: 0,
    'fixed':       lambda a: 0.2,
    'exponential': exponential(),
    'exp+jitter':  exponential_jitter(),
}

random.seed(42)
for name, strat in strategies.items():
    print(f'\n--- {name} ---')
    try:
        result, n = retry(flaky, strat, max_attempts=5, p_fail=0.7)
        print(f'  ✅ succeeded on attempt {n}')
    except Exception as e:
        print(f'  ❌ {e}')


## 📌 Takeaways

| Strategy | Good for | Risk |
|---|---|---|
| Immediate | Nothing in production | Self-DDoS |
| Fixed | Very short blips | Thundering herd |
| Exponential | Most real cases | Still synchronized |
| **Exp + jitter** | **Default choice** | None — use this |

Next up: **Notebook 2** shows *visually* why jitter matters — the thundering
herd in action — and **Notebook 3** turns this into a real-world HTTP retry
with error classification, deadlines, and retry budgets.